# A mixture model fitted with six labels

A field holds two varieties of a crop, and the seeds of each have lengths that are roughly
normal. A thousand seeds were measured, and only six were identified by hand. You will fit a
two-component Gaussian mixture by expectation-maximization (EM) using all thousand lengths,
with the six known varieties held fixed, and compare it with a model fitted to the six alone.

The model: a seed is variety $k$ with probability $w_k$, and its length then has the normal
density $\phi(x; m_k, v_k)$ with mean $m_k$ and variance $v_k$. For an identified seed the variety is known. EM alternates two
steps until the parameters settle:

- **E-step.** For each seed, the responsibilities $r_{ik}$: the probability that seed $i$ is
  variety $k$ given its length and the current parameters, or the one-hot known variety.
- **M-step.** New parameters from the responsibilities, each seed counting as $r_{ik}$ of a
  seed of variety $k$.

**How to work through it.** Part 3 asks for two functions, Parts 1 and 2 one each. Replace
each `TODO` and its `raise NotImplementedError` with your code, then run the check cell
under it. The check prints `Part N passed` when your function is right, and stops with a
message that says what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Lengths in millimeters: 600 seeds of variety 0, then 400 of variety 1.
rng = np.random.default_rng(4)
x = np.r_[rng.normal(4.0, 0.6, 600), rng.normal(6.5, 0.9, 400)]
variety = np.r_[np.zeros(600, int), np.ones(400, int)]   # for scoring only
pick = np.random.default_rng(4)
labels = np.full(len(x), -1)                             # -1: not identified
labels[pick.choice(600, 3, replace=False)] = 0
labels[600 + pick.choice(400, 3, replace=False)] = 1
print("identified seeds:", np.round(x[labels >= 0], 2), "varieties", labels[labels >= 0])

## Part 1 (coding task): The E-step

Write `e_step(x, labels, weights, means, variances)` for lengths `x` of shape `(n,)`, labels
with `-1` for an unidentified seed, and parameter arrays of shape `(K,)`. Return the
$n \times K$ responsibilities. For an unidentified seed, row $i$ is proportional to the
products $w_k\, \phi(x_i; m_k, v_k)$, scaled to sum to 1. For an identified seed it is the
one-hot row of its variety.

In [ ]:
def log_joint(x, weights, means, variances):
    """log(w_k phi(x_i; m_k, v_k)), n x K."""
    x = np.asarray(x, dtype=float)[:, None]
    return np.log(weights) - 0.5 * np.log(2 * np.pi * variances) - (x - means) ** 2 / (2 * variances)


def e_step(x, labels, weights, means, variances):
    """Responsibilities, n x K. Identified seeds get their one-hot variety."""
    # TODO
    raise NotImplementedError

In [ ]:
w0, m0, v0 = np.array([[0.5, 0.5], [0.0, 2.0], [1.0, 1.0]])   # weights, means, variances
x3, labels3 = np.array([0.0, 1.0, 0.2]), np.array([-1, -1, 1])
R = e_step(x3, labels3, w0, m0, v0)
assert R.shape == (3, 2), f"expected shape (3, 2), got {R.shape}"
assert np.allclose(R[0], [1 / (1 + np.exp(-2)), np.exp(-2) / (1 + np.exp(-2))]), f"row 0 is {R[0]}: weight each density by its w_k and normalize"
assert np.allclose(R[1], [0.5, 0.5]), f"row 1 is {R[1]}: x = 1 is halfway between equal components"
assert np.allclose(R[2], [0, 1]), f"row 2 is {R[2]}: an identified seed keeps its variety, wherever its length lies"
lone = np.array([-1])
wide = np.array([[0.25, 0.75], [0.0, 3.0], [1.0, 4.0]])
R = e_step(np.array([1.0]), lone, *wide)
a = 0.25 * np.exp(-0.5) / np.sqrt(2 * np.pi)
b = 0.75 * np.exp(-0.5) / np.sqrt(8 * np.pi)
assert np.allclose(R[0], [a / (a + b), b / (a + b)]), f"unequal weights and variances: got {R[0]}, expected {[a / (a + b), b / (a + b)]}"
far_params = np.array([[0.5, 0.5], [0.0, 3.0], [1.0, 4.0]])
R = e_step(np.array([200.0]), lone, *far_params)
assert np.all(np.isfinite(R)), f"x = 200 gives {R[0]}: both densities underflow to 0, so work with logarithms"
assert np.allclose(R[0], [0, 1]), f"x = 200 should belong to the wider component, got {R[0]}"
print("Part 1 passed")

## Part 2 (coding task): The M-step

Write `m_step(x, R)` returning `(weights, means, variances)` from the responsibilities.
Write $N_k$ for the total responsibility of variety $k$, summed over all $n$ seeds. Then
the new weight is $w_k = N_k / n$, the new mean $m_k$ is the average of the lengths with seed
$i$ weighted by $r_{ik}$, and the new variance $v_k$ is the average of $(x_i - m_k)^2$ with
the same weights:

$$
m_k = \textstyle\sum_i r_{ik}\, x_i \,/\, N_k , \qquad v_k = \textstyle\sum_i r_{ik} (x_i - m_k)^2 \,/\, N_k .
$$

In [ ]:
def m_step(x, R):
    """Mixture weights, means and variances that maximize the expected log-likelihood."""
    # TODO
    raise NotImplementedError

In [ ]:
xs = np.array([0.0, 1.0, 2.0, 4.0])
Rs = np.array([[1.0, 0.0], [0.5, 0.5], [0.25, 0.75], [0.0, 1.0]])
w, m, v = m_step(xs, Rs)
assert np.allclose(w, [1.75 / 4, 2.25 / 4]), f"weights {w}: w_k is the share of the n seeds' responsibility"
assert np.allclose(m, [1 / 1.75, 6 / 2.25]), f"means {m}: divide the weighted sum by N_k, not by n"
assert np.allclose(v, [26 / 49, 14 / 9]), f"variances {v}, expected [26/49, 14/9]: weight each squared deviation from the new mean by r_ik"
print("Part 2 passed")

## Part 3 (coding task): The log-likelihood and the loop

Write two functions.

- `log_likelihood(x, labels, weights, means, variances)` returns the log-probability of
  every length and every known variety. An unidentified seed contributes the log of its
  mixture density, $\log\big(w_0\, \phi(x_i; m_0, v_0) + w_1\, \phi(x_i; m_1, v_1)\big)$. An
  identified seed of variety $y_i$ contributes $\log\big(w_{y_i}\, \phi(x_i; m_{y_i}, v_{y_i})\big)$.
- `fit(x, labels, n_iter)` starts from the six identified seeds alone: the weights are their
  variety shares, and the mean and variance of each variety are those of its identified
  seeds (dividing by the count, not the count minus one). Then it runs `n_iter` rounds of
  E-step and M-step and returns `(weights, means, variances, history)`, where `history`
  lists the log-likelihood after each round.

The check fits the seeds and compares the result with the model fitted to the six alone.

In [ ]:
def log_likelihood(x, labels, weights, means, variances):
    # TODO
    raise NotImplementedError


def fit(x, labels, n_iter=200):
    # TODO
    raise NotImplementedError

In [ ]:
w0, m0, v0 = np.array([[0.5, 0.5], [0.0, 2.0], [1.0, 1.0]])
x3, labels3 = np.array([0.0, 1.0, 0.2]), np.array([-1, -1, 1])
lone = np.array([-1])
far_params = np.array([[0.5, 0.5], [0.0, 3.0], [1.0, 4.0]])
ll = log_likelihood(x3, labels3, w0, m0, v0)
c = -0.5 * np.log(2 * np.pi)
expected = (np.log(0.5 * np.exp(c) + 0.5 * np.exp(c - 2))
            + np.log(np.exp(c - 0.5))
            + np.log(0.5) + c - 1.8 ** 2 / 2)
assert np.isclose(ll, expected), f"log-likelihood {ll}, expected {expected}: an identified seed counts log(w_y phi(x; m_y, v_y)), not the mixture"
far = log_likelihood(np.array([200.0]), lone, *far_params)
assert np.isfinite(far), "the log-likelihood of x = 200 should be a large negative number, not -inf"

known = labels >= 0
w6 = np.bincount(labels[known], minlength=2) / 6
m6 = np.array([x[labels == k].mean() for k in range(2)])
v6 = np.array([x[labels == k].var() for k in range(2)])
weights, means, variances, history = fit(x, labels, 200)
assert len(history) == 200, f"expected 200 log-likelihood values, got {len(history)}"
assert np.all(np.diff(history) > -1e-8), "EM never lowers the log-likelihood: check the E-step and M-step together"
assert np.allclose(means, [4.0040, 6.4544], atol=2e-3) and np.allclose(variances, [0.3672, 0.8606], atol=2e-3), f"fitted means {means}, variances {variances}"
assert np.allclose(e_step(x, labels, weights, means, variances)[known], np.eye(2)[labels[known]]), "identified seeds must keep their varieties"
predict = lambda w, m, v: log_joint(x, w, m, v).argmax(axis=1)
acc6 = np.mean(predict(w6, m6, v6)[~known] == variety[~known])
acc = np.mean(predict(weights, means, variances)[~known] == variety[~known])
print(f"six seeds alone: weights {np.round(w6, 3)}, means {np.round(m6, 2)}, variances {np.round(v6, 3)}, accuracy {acc6:.3f}")
print(f"EM, all seeds:   weights {np.round(weights, 3)}, means {np.round(means, 2)}, variances {np.round(variances, 3)}, accuracy {acc:.3f}")
assert acc > acc6 + 0.1, "EM on all the seeds should beat the six-seed model by a wide margin"
print("Part 3 passed")


## Solution

### Part 1: The E-step

In [ ]:
def log_joint(x, weights, means, variances):
    """log(w_k phi(x_i; m_k, v_k)), n x K."""
    x = np.asarray(x, dtype=float)[:, None]
    return np.log(weights) - 0.5 * np.log(2 * np.pi * variances) - (x - means) ** 2 / (2 * variances)


def e_step(x, labels, weights, means, variances):
    """Responsibilities, n x K. Identified seeds get their one-hot variety."""
    L = log_joint(x, weights, means, variances)
    L = L - L.max(axis=1, keepdims=True)             # the largest entry of each row becomes 0
    R = np.exp(L)
    R /= R.sum(axis=1, keepdims=True)
    known = labels >= 0
    R[known] = np.eye(len(weights))[labels[known]]
    return R

**Why it works.** At $x = 200$ both densities are below the smallest positive double, so
computing them directly gives $0/0$. In logarithms the ratio is untouched. Subtracting each
row's largest log term before exponentiating turns that term into exactly 1, so every row
has a denominator of 1 or more and no division by zero can happen. The constant cancels in the normalization. An
identified seed's responsibilities are its label, which is how the six known varieties
enter every round of EM.

### Part 2: The M-step

In [ ]:
def m_step(x, R):
    """Mixture weights, means and variances that maximize the expected log-likelihood."""
    x = np.asarray(x, dtype=float)
    Nk = R.sum(axis=0)
    weights = Nk / len(x)
    means = R.T @ x / Nk
    variances = (R * (x[:, None] - means) ** 2).sum(axis=0) / Nk
    return weights, means, variances

**Why it works.** Each formula is the maximum-likelihood estimate for one normal distribution,
with seed $i$ counted $r_{ik}$ times. Setting the derivative of the weighted log-density
with respect to $m_k$ to zero gives the weighted mean, and then the derivative with respect
to $v_k$ gives the weighted variance about it. Maximizing the weighted log of the weights,
with the weights summing to 1, gives $N_k / n$. Identified seeds have one-hot rows, so each counts
fully toward its own variety.

### Part 3: The log-likelihood and the loop

In [ ]:
def log_likelihood(x, labels, weights, means, variances):
    L = log_joint(x, weights, means, variances)
    known = labels >= 0
    top = L[~known].max(axis=1)
    mixed = top + np.log(np.exp(L[~known] - top[:, None]).sum(axis=1))   # log-sum-exp per row
    return float(mixed.sum() + L[known, labels[known]].sum())


def fit(x, labels, n_iter=200):
    known = labels >= 0
    K = labels.max() + 1
    weights = np.bincount(labels[known], minlength=K) / known.sum()
    means = np.array([x[labels == k].mean() for k in range(K)])
    variances = np.array([x[labels == k].var() for k in range(K)])
    history = []
    for _ in range(n_iter):
        R = e_step(x, labels, weights, means, variances)
        weights, means, variances = m_step(x, R)
        history.append(log_likelihood(x, labels, weights, means, variances))
    return weights, means, variances, history

**Why it works.** The identified seeds contribute the joint probability of their length and
their variety, which is exactly what the clamped E-step assumes, so EM's guarantee holds:
each round can only raise this log-likelihood, and the check confirms it. Six seeds give
two variances from three points each. On this draw variety 0's three seeds lie within
about 0.6 mm of each other and variety 1's spread from 4.3 to 7.5 mm, so the six-seed model
calls a seed variety 0 only between about 3.1 and 4.4 mm and gives 195 of the 597
unidentified variety 0 seeds to variety 1.
EM uses the shape of all thousand lengths to recover weights near 0.6 and 0.4, means near
4.0 and 6.5 and variances near the true 0.36 and 0.81. The cluster assumption is built into
the model here: each variety is one bump of the density.